# M4 Assignment 3 Part C: Miulti-Agentic System using ollama gemma 3:4b

STEP01: Define Custom Agents (agents.py)

In [62]:
import pandas as pd
import numpy as np
import os
df = pd.read_csv("hitl_green_100.csv")


In [41]:
print(len(df))

100


In [53]:
print(df.columns)

Index(['doc_id', 'text', 'p_green', 'u', 'llm_green_suggested',
       'llm_confidence', 'llm_rationale', 'is_green_human', 'notes'],
      dtype='object')


In [42]:
from crewai import Agent, LLM
from textwrap import dedent

Required Agents:

Agent 1 (The Advocate): Argues for the green classification (Y02). // set the tempurature of 0.6 to support the decision

Agent 2 (The Skeptic): Argues against it (looks for greenwashing). // set the tempurature of 0.6 to support the decision

Agent 3 (The Judge): Weighs the arguments and produces a final JSON label + rationale. // set the tempurature of 0.3 to narrow the way of llm decision making and stability

In [43]:
class CustomAgents:
    def __init__(self):
        self.debate_llm = LLM(model="ollama/gemma3:4b", base_url="http://localhost:11434", temperature=0.6) #advocate and skeptic
        self.judge_llm = LLM(model="ollama/gemma3:4b", base_url="http://localhost:11434", temperature=0.1) #judge
    def advocate_agent(self):
        return Agent(role="Green Technology Advocate", backstory="Expert in green environmental innovation and CPC Y02 classification. Understand the benefits of green technology and its impact on sustainability and the environment.",
                     goal="Argue why the patent should be classified under CPC Y02 and highlight the advantages of green technology in the context of the patent.", llm=self.debate_llm, verbose=False)
    def skeptic_agent(self):
        return Agent(role="Greenwashing Auditor", backstory="Critical thinker who questions the classification of patents under CPC Y02. Focuses on identifying potential weaknesses in the argument for green technology classification.",
                     goal="Challenge and Argue the advocate's arguments for classifying the patent under CPC Y02 and provide counterpoints to highlight any potential shortcomings or alternative classifications.", llm=self.debate_llm, verbose=False) 
    def judge_agent(self):
        return Agent(role="Experienced Patent Classification Judge", backstory="Objective evaluator responsible for assessing the arguments presented by both the advocate and skeptic. Focuses on determining the validity of the classification under CPC Y02 based on the strength of the arguments and evidence provided.",
                     goal="Evaluate the arguments from both the advocate and skeptic, and make a final decision on whether the patent should be classified under CPC Y02 based on the merits of the arguments presented. Return final decision in strict JSON format", llm=self.judge_llm, verbose=False)
    

STEP02: Define Custom Tasks(task.py)

In [44]:
from crewai import Task

In [45]:
class CustomTask:
    def advocate_task(self, agent, claim):
        return Task(description=dedent(f"""Patent Claim: {claim}\n\n As the Green Technology Advocate, your task is to argue why this patent should be classified under CPC Y02. Highlight the advantages of green technology in the context of this patent and provide strong arguments to support your position."""),
                    expected_output=f"Arguments supporting the classification of the patent under CPC Y02: {claim}", agent=agent)
    def skeptic_task(self, agent, claim):
        return Task(description=dedent(f"""Patent Claim: {claim}\n\n As the Greenwashing Auditor, your task is to challenge the advocate's arguments for classifying this patent under CPC Y02. Provide counterpoints to highlight any potential shortcomings or alternative classifications."""),
                    expected_output=f"Counterarguments challenging the classification of the patent under CPC Y02: {claim}", agent=agent)
    def judge_task(self, agent, claim):
        return Task(description=dedent(f"""Patent Claim: {claim} \n\n As the Experienced Patent Classification Judge, your task is to evaluate the arguments from both the advocate and skeptic, and make a final decision on whether the patent should be classified under CPC Y02 based on the merits of the arguments presented. 
                                       Return final decision in strict JSON format:
                                       {{"advocate_argument": "short summary of advocate position",
                                       "skeptic_argument": "short summary of skeptic position",
                                        "is_green_tech": "0" or "1" which 1 indicating whether the patent should be classified under CPC Y02,
                                           "judge_rationale": "2-3 sentence of detailed explanation of the decision based on the arguments presented."
                                       }} Rules - No Markdown - No explaination outside JSON - Only return valid JSON"""),
                    expected_output=f"Strict JSON output following the specified format", agent=agent)

STEP03: Run locally and save result

In [46]:
import crewai
print(crewai.__version__)

1.9.3


In [47]:
import pandas as pd
import json
import logging
import time
from crewai import Crew, Task, Agent
import re


In [48]:
from crewai import LLM

llm = LLM(
    model="ollama/gemma3:4b",
    base_url="http://localhost:11434",
    temperature=0.3
)

response = llm.call("Say hello in one sentence.")
print(response)

Hello there! 😊


Test running

In [50]:
agents = CustomAgents()
tasks = CustomTask()

In [51]:
# RUN SECTION

claim = "A system for reducing energy waste in industrial heating processes."

agents = CustomAgents()
tasks = CustomTask()

advocate = agents.advocate_agent()
skeptic = agents.skeptic_agent()
judge = agents.judge_agent()

t1 = tasks.advocate_task(advocate, claim)
t2 = tasks.skeptic_task(skeptic, claim)
t3 = tasks.judge_task(judge, claim)

crew = Crew(
    agents=[advocate, skeptic, judge],
    tasks=[t1, t2, t3],
    verbose=False
)

output = crew.kickoff()

raw = str(output)
print(raw)

cleaned = re.sub(r"```json|```", "", raw).strip()
json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)
if json_match:
    parsed = json.loads(json_match.group())

    advocate_argument = parsed["advocate_argument"]
    skeptic_argument = parsed["skeptic_argument"]
    is_green_tech = int(parsed["is_green_tech"])
    judge_rationale = parsed["judge_rationale"]

    print("Prediction:", is_green_tech)
else:
    print("No JSON found")

```json
{
  "advocate_argument": "To the Patent Examiner, Regarding the patent claim: ‘A system for reducing energy waste in industrial heating processes,’ I unequivocally recommend classification under CPC Y02. This system represents a core application of green technology and aligns perfectly with the spirit and scope of this classification. My argument rests on several key points, demonstrating the substantial environmental and sustainability benefits inherent in this innovation. The system directly reduces energy consumption & carbon footprint, leverages green technology principles like process optimization and waste heat recovery, and aligns with international sustainability goals. It’s a system built on the principles of green technology, directly addressing a critical environmental challenge.",
  "skeptic_argument": "To the Patent Examiner, I must respectfully challenge the advocate’s unequivocal recommendation of CPC Y02. While the system’s intent – reducing energy waste – is la

Run 100 claims + suggested by chatGPT after so many errors due to version of crewAI

In [52]:
import json
import re
import pandas as pd
from crewai import Crew

results = []

for idx, row in df.iterrows():

    claim = row["text"]
    doc_id = row["doc_id"]

    print(f"\nProcessing {idx+1}/{len(df)} | doc_id={doc_id}")

    try:
        # สร้าง agents ใหม่
        agents = CustomAgents()
        tasks = CustomTask()

        advocate = agents.advocate_agent()
        skeptic = agents.skeptic_agent()
        judge = agents.judge_agent()

        t1 = tasks.advocate_task(advocate, claim)
        t2 = tasks.skeptic_task(skeptic, claim)
        t3 = tasks.judge_task(judge, claim)

        crew = Crew(
            agents=[advocate, skeptic, judge],
            tasks=[t1, t2, t3],
            verbose=False
        )

        output = crew.kickoff()
        raw = str(output)

        # --------------------------
        # CLEAN JSON BLOCK
        # --------------------------

        cleaned = re.sub(r"```json|```", "", raw).strip()

        json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)

        if not json_match:
            print(f"❌ No JSON found | doc_id={doc_id}")
            continue

        json_string = json_match.group()

        # remove invalid control characters
        json_string = re.sub(r'[\x00-\x1f\x7f]', '', json_string)

        # check if truncated
        if not json_string.strip().endswith("}"):
            print(f"⚠ Incomplete JSON | doc_id={doc_id}")
            continue

        try:
            parsed = json.loads(json_string)
        except json.JSONDecodeError:
            print(f"❌ JSON decode failed | doc_id={doc_id}")
            print(json_string[:300])
            continue

        # --------------------------
        # SAFE EXTRACTION
        # --------------------------

        results.append({
            "doc_id": doc_id,
            "advocate_argument": parsed.get("advocate_argument", ""),
            "skeptic_argument": parsed.get("skeptic_argument", ""),
            "is_green_tech": int(parsed.get("is_green_tech", 0)),
            "judge_rationale": parsed.get("judge_rationale", "")
        })

        print("✅ Done")

    except Exception as e:
        print(f"🔥 Unexpected error at doc_id={doc_id}: {e}")
        continue


# --------------------------
# SAVE RESULT
# --------------------------

results_df = pd.DataFrame(results)
results_df.to_csv("A3_agent_labels_100.csv", index=False)

print("\n🎉 Finished all claims")


Processing 1/100 | doc_id=9315787
✅ Done

Processing 2/100 | doc_id=9334087
✅ Done

Processing 3/100 | doc_id=9686855
✅ Done

Processing 4/100 | doc_id=9551706
✅ Done

Processing 5/100 | doc_id=9745777
✅ Done

Processing 6/100 | doc_id=9237703
✅ Done

Processing 7/100 | doc_id=9273641
✅ Done

Processing 8/100 | doc_id=9206804
✅ Done

Processing 9/100 | doc_id=8466241
✅ Done

Processing 10/100 | doc_id=8581146
✅ Done

Processing 11/100 | doc_id=8975201
✅ Done

Processing 12/100 | doc_id=9732596
✅ Done

Processing 13/100 | doc_id=8661630
✅ Done

Processing 14/100 | doc_id=9484843
✅ Done

Processing 15/100 | doc_id=9703511
✅ Done

Processing 16/100 | doc_id=8770118
✅ Done

Processing 17/100 | doc_id=9485853
✅ Done

Processing 18/100 | doc_id=8928526
✅ Done

Processing 19/100 | doc_id=8607927
✅ Done

Processing 20/100 | doc_id=9449733
✅ Done

Processing 21/100 | doc_id=9683299
✅ Done

Processing 22/100 | doc_id=9510426
✅ Done

Processing 23/100 | doc_id=9787709
✅ Done

Processing 24/100 |

There are 2 claims that JSON decode fail, so i find the missing one from the original 100 high risk, and then run small parse for those 2 with minimal version. After that I put it back to dataframe and then save again in final version of this debate agent 100 claims.

In [59]:
import pandas as pd
df_A3 = pd.read_csv("A3_agent_labels_100.csv")
print(len(df))

98


In [60]:
results_df = pd.read_csv("A3_agent_labels_100.csv")
print(len(results_df))  # ควรได้ 98


98


In [63]:
missing_ids = [9777344, 9485858]
missing_claims = df[df["doc_id"].isin(missing_ids)]
missing_claims

,doc_id,text,p_green,u,llm_green_suggested,llm_confidence,llm_rationale,is_green_human,notes
88,9485858,1. A flexible display device comprising: a fle...,0.493586,0.987171,NaN,NaN,NaN,NaN,NaN
99,9777344,1. A stainless steel having superior surface q...,0.492495,0.984991,NaN,NaN,NaN,NaN,NaN


In [64]:
for _, row in missing_claims.iterrows():

    claim = row["text"]
    doc_id = row["doc_id"]

    agents = CustomAgents()
    tasks = CustomTask()

    advocate = agents.advocate_agent()
    skeptic = agents.skeptic_agent()
    judge = agents.judge_agent()

    t1 = tasks.advocate_task(advocate, claim)
    t2 = tasks.skeptic_task(skeptic, claim)
    t3 = tasks.judge_task(judge, claim)

    crew = Crew(
        agents=[advocate, skeptic, judge],
        tasks=[t1, t2, t3],
        verbose=False
    )

    output = crew.kickoff()
    raw = str(output)

    cleaned = re.sub(r"```json|```", "", raw).strip()
    json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)

    if not json_match:
        print(f"Still failed for {doc_id}")
        continue

    parsed = json.loads(json_match.group())

    new_row = {
        "doc_id": doc_id,
        "advocate_argument": parsed.get("advocate_argument", ""),
        "skeptic_argument": parsed.get("skeptic_argument", ""),
        "is_green_tech": int(parsed.get("is_green_tech", 0)),
        "judge_rationale": parsed.get("judge_rationale", "")
    }

    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

In [65]:
print(len(results_df))

100


In [66]:
results_df.to_csv("A3_agent_labels_100_FINAL.csv", index=False)

In [67]:
results_df = pd.read_csv("A3_agent_labels_100_FINAL.csv")

In [68]:
pd.set_option("display.max_colwidth", None)

print(results_df.head(3))

    doc_id  \
0  9315787   
1  9334087   
2  9686855   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     advocate_argument  \
0  The DNA polymerase patent represents a significant step forward in green biotechnology – a move toward precision, efficiency, and reduced environmental impact. The targeted amino acid modifications, combined with the high degree of similarity to SEQ ID NO:38, demonstrate a deliberate effort to optimize an existing enzyme, reducing resou

See 3 examples from agents debate result

In [69]:
for i in range(3):
    row = results_df.iloc[i]
    
    print("="*80)
    print(f"DOC_ID: {row['doc_id']}")
    print("\nADVOCATE:")
    print(row['advocate_argument'])
    print("\nSKEPTIC:")
    print(row['skeptic_argument'])
    print("\nPREDICTION:", row['is_green_tech'])
    print("\nJUDGE RATIONALE:")
    print(row['judge_rationale'])
    print("="*80)
    print("\n\n")

DOC_ID: 9315787

ADVOCATE:
The DNA polymerase patent represents a significant step forward in green biotechnology – a move toward precision, efficiency, and reduced environmental impact. The targeted amino acid modifications, combined with the high degree of similarity to SEQ ID NO:38, demonstrate a deliberate effort to optimize an existing enzyme, reducing resource consumption and minimizing the risk of unintended consequences. This aligns perfectly with the core principles of CPC Y02, particularly its focus on biotechnological processes that minimize environmental impact and promote sustainable enzyme efficiency.

SKEPTIC:
Classifying this DNA polymerase solely under CPC Y02 risks overstating its green credentials. The claim’s reliance on a pre-existing, naturally occurring enzyme, coupled with the ‘at least 95% identical’ specification, suggests a refinement rather than a revolutionary advancement. The focus on optimization doesn’t inherently guarantee a reduced environmental impact